# Medição de borrachas com FastSAM + ArUco (GPU)

Roda o pipeline do repositório [`JeanJuba/sam-object-recognition-measurement`](https://github.com/JeanJuba/sam-object-recognition-measurement) no Colab com GPU.

**Antes de começar:** menu *Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU*.

Depois é só executar as células em ordem. Diferenças em relação a rodar no PC:
- usa o **FastSAM-x** (contornos melhores) com `frame_stride: 1` (todos os frames) — na T4 isso leva poucos minutos;
- no final o vídeo é recomprimido em H.264 (arquivo leve) e exibido aqui no notebook.

In [ ]:
# 1) Confere se a GPU está ativa (deve listar uma Tesla T4)
!nvidia-smi

In [ ]:
# 2) Baixa o código do projeto
import os
%cd /content
if not os.path.exists('sam-object-recognition-measurement'):
    !git clone https://github.com/JeanJuba/sam-object-recognition-measurement.git
%cd sam-object-recognition-measurement
!git log --oneline -1

In [ ]:
# 3) Instala as dependências (o Colab já vem com PyTorch + CUDA)
%pip install -q -r requirements.txt

In [ ]:
# 4) Garante o vídeo: usa o do repositório se existir; senão pede upload
import os
video = 'data/videos/aruco_longo.mp4'
if not os.path.exists(video):
    from google.colab import files
    print('Selecione o arquivo aruco_longo.mp4 do seu computador:')
    up = files.upload()
    os.makedirs('data/videos', exist_ok=True)
    os.replace(next(iter(up)), video)
print(f'OK: {video} ({os.path.getsize(video)/1e6:.1f} MB)')

In [ ]:
# 5) Cria o perfil de config para GPU a partir do config_aruco_longo.yaml:
#    modelo maior (FastSAM-x) e todos os frames (stride 1)
import yaml
with open('config_aruco_longo.yaml', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
cfg['segmentation']['model'] = 'FastSAM-x.pt'
cfg['video']['frame_stride'] = 1
with open('config_colab.yaml', 'w', encoding='utf-8') as f:
    yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)
print('config_colab.yaml criado:', cfg['segmentation']['model'],
      '| stride', cfg['video']['frame_stride'])

In [ ]:
# 6) Roda o pipeline completo (calibração ArUco -> FastSAM -> medições)
#    Na T4: ~3-5 min para o vídeo de 60 s
!python main.py --config config_colab.yaml run

In [ ]:
# 7) Tabela de medições por peça
import pandas as pd
pd.read_csv('data/output/medicoes.csv')

In [ ]:
# 8) Recomprime o vídeo anotado em H.264 (arquivo leve) e gera uma prévia
!ffmpeg -y -loglevel error -i data/output/video_anotado.mp4 \
    -c:v libx264 -crf 23 -preset veryfast -movflags +faststart \
    data/output/video_anotado_h264.mp4
!ffmpeg -y -loglevel error -i data/output/video_anotado.mp4 \
    -c:v libx264 -crf 28 -preset veryfast -vf scale=480:-2 -movflags +faststart \
    data/output/preview.mp4
import os
print(f"final:  {os.path.getsize('data/output/video_anotado_h264.mp4')/1e6:.1f} MB")
print(f"previa: {os.path.getsize('data/output/preview.mp4')/1e6:.1f} MB")

In [ ]:
# 9) Exibe a prévia aqui no notebook
import base64
from IPython.display import HTML
with open('data/output/preview.mp4', 'rb') as f:
    data = base64.b64encode(f.read()).decode()
HTML(f'<video controls width="360" src="data:video/mp4;base64,{data}"></video>')

In [ ]:
# 10) Baixa os resultados para o seu computador
from google.colab import files
files.download('data/output/video_anotado_h264.mp4')
files.download('data/output/medicoes.csv')